In [ ]:
# =====================================================================
# IEEE ACCESS R1 — PHASE 3
# DEPENDENCY-AWARE STATISTICAL INFERENCE
# GLOBAL COHORT
#
# PURPOSE
# -------
# Address Reviewer 1 Comment 3:
#
# The original paired bootstrap treats individual match observations
# as independent, but teams recur across matches and observations are
# clustered by competition and chronological periods.
#
# This sensitivity analysis therefore repeats inference using WHOLE
# dependency blocks rather than individual matches.
#
# PRIMARY DEPENDENCY-AWARE ANALYSIS
# ---------------------------------
# Competition-season cluster bootstrap.
#
# SECONDARY COARSER SENSITIVITY
# -----------------------------
# Competition-level cluster bootstrap.
#
# REFERENCE ONLY
# --------------
# Ordinary match-level paired bootstrap.
#
# NO MODEL IS RETRAINED.
# NO PREDICTION IS CHANGED.
# NO HYPERPARAMETER IS TUNED.
#
# Frozen input:
# Phase 2B combined Global OOF predictions
# N = 1745 paired observations
#
# Expected competition-season clusters = 20
# Expected competition clusters        = 11
#
# Main comparisons:
# 1. C-Hybrid-full vs A-MLP
# 2. C-Hybrid-full vs TF-Hybrid
# 3. TF-Hybrid vs A-MLP
# 4. B-GNN-full vs TF-NN
#
# DO NOT ALTER THE BLOCK DEFINITION AFTER SEEING RESULTS.
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

import os
import json
import math
import shutil
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# =====================================================================
# 1. FROZEN SETTINGS
# =====================================================================

N_BOOTSTRAP = 10_000

SEED_MATCH_BOOTSTRAP = 7311
SEED_COMP_SEASON_BOOTSTRAP = 7312
SEED_COMPETITION_BOOTSTRAP = 7313

EXPECTED_PAIRED_N = 1745
EXPECTED_MODEL_COUNT = 5
EXPECTED_COMP_SEASON_BLOCKS = 20
EXPECTED_COMPETITION_BLOCKS = 11

EXPECTED_COMBINED_PREDICTIONS_SHA256 = (
    "d5df0e00932a375dc7faa998778c85d9"
    "44436d43267257c269a225355c689aab"
)

EXPECTED_INPUT_NAME = (
    "IEEE_Access_R1_Exp6_Topology_Free_GLOBAL.zip"
)

OUTPUT_ROOT = Path(
    "/content/ieee_access_phase3_dependency"
)

FINAL_ZIP = Path(
    "/content/"
    "IEEE_Access_R1_Phase3_Dependency_Aware_GLOBAL.zip"
)

COMPARISONS = [
    (
        "C-Hybrid-full",
        "A-MLP",
        "C-Hybrid-full vs A-MLP",
        "original_main_global_comparison",
    ),
    (
        "C-Hybrid-full",
        "TF-Hybrid",
        "C-Hybrid-full vs TF-Hybrid",
        "graph_specific_comparison",
    ),
    (
        "TF-Hybrid",
        "A-MLP",
        "TF-Hybrid vs A-MLP",
        "topology_free_complementarity",
    ),
    (
        "B-GNN-full",
        "TF-NN",
        "B-GNN-full vs TF-NN",
        "standalone_graph_specific_comparison",
    ),
]


print("=" * 100)
print("IEEE ACCESS R1 — PHASE 3")
print("DEPENDENCY-AWARE GLOBAL STATISTICAL INFERENCE")
print("=" * 100)

print(
    "\nNo GPU is required. "
    "This phase uses frozen OOF predictions only."
)


# =====================================================================
# 2. FIND OR UPLOAD PHASE 2B ZIP
# =====================================================================

input_zip = Path(
    "/content"
) / EXPECTED_INPUT_NAME


if not input_zip.exists():

    print(
        "\nPhase 2B ZIP was not found in /content/."
    )

    print(
        "Please upload:"
    )

    print(
        EXPECTED_INPUT_NAME
    )

    try:
        from google.colab import files

        uploaded = files.upload()

        if not uploaded:
            raise RuntimeError(
                "No file was uploaded."
            )

        matching = [
            filename
            for filename
            in uploaded.keys()
            if filename.endswith(".zip")
        ]

        if not matching:
            raise RuntimeError(
                "No ZIP file was uploaded."
            )

        # Prefer exact expected filename.
        if EXPECTED_INPUT_NAME in uploaded:

            input_zip = (
                Path("/content")
                / EXPECTED_INPUT_NAME
            )

        else:

            input_zip = (
                Path("/content")
                / matching[0]
            )

    except Exception as exc:

        raise RuntimeError(
            "Could not obtain Phase 2B ZIP.\n"
            "Upload the ZIP into /content/ and rerun."
        ) from exc


print(
    "\nInput ZIP:",
    input_zip
)


# =====================================================================
# 3. EXTRACT PHASE 2B ARTIFACT
# =====================================================================

if OUTPUT_ROOT.exists():
    shutil.rmtree(
        OUTPUT_ROOT
    )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXTRACT_DIR = (
    OUTPUT_ROOT
    / "phase2b_input"
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


with zipfile.ZipFile(
    input_zip,
    "r",
) as archive:

    archive.extractall(
        EXTRACT_DIR
    )


combined_path = (
    EXTRACT_DIR
    / "outputs"
    / "combined_global_oof_predictions.csv"
)


if not combined_path.exists():

    raise RuntimeError(
        "Could not find:\n"
        "outputs/combined_global_oof_predictions.csv\n"
        "inside the uploaded Phase 2B ZIP."
    )


# =====================================================================
# 4. VERIFY EXACT FROZEN INPUT ARTIFACT
# =====================================================================

def sha256_file(
    path: Path,
) -> str:

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        for chunk in iter(
            lambda:
            handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


combined_sha = sha256_file(
    combined_path
)


print(
    "\nCombined prediction SHA-256:"
)

print(
    combined_sha
)


if (
    combined_sha
    != EXPECTED_COMBINED_PREDICTIONS_SHA256
):

    raise RuntimeError(
        "\nINPUT ARTIFACT HASH MISMATCH.\n"
        "The uploaded predictions are not the exact "
        "Phase 2B artifact previously inspected.\n\n"
        f"Expected:\n"
        f"{EXPECTED_COMBINED_PREDICTIONS_SHA256}\n\n"
        f"Actual:\n"
        f"{combined_sha}\n"
    )


print(
    "✓ Exact frozen Phase 2B predictions verified."
)


# =====================================================================
# 5. LOAD PREDICTIONS
# =====================================================================

predictions = pd.read_csv(
    combined_path
)


required_columns = {
    "match_id",
    "competition_id",
    "competition_name",
    "season_id",
    "fold",
    "model",
    "true_label",
    "predicted_class",
    "predicted_score",
}


missing_columns = (
    required_columns
    - set(
        predictions.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "Required columns are missing:\n"
        + str(
            sorted(
                missing_columns
            )
        )
    )


predictions[
    "match_id"
] = predictions[
    "match_id"
].astype(
    int
)

predictions[
    "true_label"
] = predictions[
    "true_label"
].astype(
    int
)

predictions[
    "predicted_class"
] = predictions[
    "predicted_class"
].astype(
    int
)


expected_models = {
    "A-MLP",
    "B-GNN-full",
    "TF-NN",
    "C-Hybrid-full",
    "TF-Hybrid",
}


actual_models = set(
    predictions[
        "model"
    ].unique()
)


if actual_models != expected_models:

    raise RuntimeError(
        "Unexpected model set.\n\n"
        f"Expected: {sorted(expected_models)}\n"
        f"Actual:   {sorted(actual_models)}"
    )


# =====================================================================
# 6. VERIFY EXACT PAIRED OOF MATCH SET
# =====================================================================

model_match_sets = {}

for model_name in sorted(
    expected_models
):

    block = predictions[
        predictions[
            "model"
        ].eq(
            model_name
        )
    ]

    if block.duplicated(
        ["match_id"]
    ).any():

        raise RuntimeError(
            f"Duplicate match IDs found for {model_name}."
        )

    model_match_sets[
        model_name
    ] = set(
        block[
            "match_id"
        ].astype(
            int
        )
    )


reference_ids = (
    model_match_sets[
        "C-Hybrid-full"
    ]
)


for model_name, ids in model_match_sets.items():

    if ids != reference_ids:

        raise RuntimeError(
            f"Paired OOF IDs differ for {model_name}."
        )


if len(
    reference_ids
) != EXPECTED_PAIRED_N:

    raise RuntimeError(
        "Paired Global OOF N mismatch.\n"
        f"Expected: {EXPECTED_PAIRED_N}\n"
        f"Actual:   {len(reference_ids)}"
    )


print(
    "\n✓ Exact paired OOF set verified."
)

print(
    "Paired N:",
    len(
        reference_ids
    )
)


# =====================================================================
# 7. BUILD ONE CANONICAL ROW PER MATCH
# =====================================================================

reference_model = (
    predictions[
        predictions[
            "model"
        ].eq(
            "C-Hybrid-full"
        )
    ]
    .sort_values(
        "match_id"
    )
    .reset_index(
        drop=True
    )
)


metadata = reference_model[
    [
        "match_id",
        "competition_id",
        "competition_name",
        "season_id",
        "fold",
        "true_label",
    ]
].copy()


if metadata[
    [
        "competition_id",
        "season_id",
    ]
].isna().any().any():

    raise RuntimeError(
        "Competition or season IDs contain missing values."
    )


# Use integer IDs to define blocks.
metadata[
    "competition_id"
] = metadata[
    "competition_id"
].astype(
    int
)

metadata[
    "season_id"
] = metadata[
    "season_id"
].astype(
    int
)


metadata[
    "competition_season_block"
] = (
    metadata[
        "competition_id"
    ].astype(
        str
    )
    + "__"
    + metadata[
        "season_id"
    ].astype(
        str
    )
)


metadata[
    "competition_block"
] = (
    metadata[
        "competition_id"
    ].astype(
        str
    )
)


n_comp_season_blocks = (
    metadata[
        "competition_season_block"
    ].nunique()
)

n_competition_blocks = (
    metadata[
        "competition_block"
    ].nunique()
)


print(
    "\nCompetition-season blocks:",
    n_comp_season_blocks
)

print(
    "Competition blocks:",
    n_competition_blocks
)


if (
    n_comp_season_blocks
    != EXPECTED_COMP_SEASON_BLOCKS
):

    raise RuntimeError(
        "Unexpected competition-season block count.\n"
        f"Expected: {EXPECTED_COMP_SEASON_BLOCKS}\n"
        f"Actual:   {n_comp_season_blocks}"
    )


if (
    n_competition_blocks
    != EXPECTED_COMPETITION_BLOCKS
):

    raise RuntimeError(
        "Unexpected competition block count.\n"
        f"Expected: {EXPECTED_COMPETITION_BLOCKS}\n"
        f"Actual:   {n_competition_blocks}"
    )


print(
    "✓ Dependency block structure verified."
)


# =====================================================================
# 8. MERGE EACH MODEL'S PREDICTIONS INTO CANONICAL MATCH TABLE
# =====================================================================

wide = metadata.copy()


for model_name in sorted(
    expected_models
):

    model_block = (
        predictions[
            predictions[
                "model"
            ].eq(
                model_name
            )
        ]
        [
            [
                "match_id",
                "predicted_class",
                "predicted_score",
            ]
        ]
        .copy()
    )

    safe_name = (
        model_name
        .replace(
            "-",
            "_"
        )
    )

    model_block = (
        model_block
        .rename(
            columns={
                "predicted_class":
                    f"pred_{safe_name}",

                "predicted_score":
                    f"score_{safe_name}",
            }
        )
    )

    wide = wide.merge(
        model_block,
        on="match_id",
        how="left",
        validate="one_to_one",
    )


if wide.isna().any().any():

    missing_cols = wide.columns[
        wide.isna().any()
    ].tolist()

    raise RuntimeError(
        "Unexpected missing values after pairing:\n"
        + str(
            missing_cols
        )
    )


# =====================================================================
# 9. METRIC HELPERS FROM CONFUSION COUNTS
# =====================================================================

def confusion_counts(
    truth,
    prediction,
):
    truth = np.asarray(
        truth,
        dtype=int,
    )

    prediction = np.asarray(
        prediction,
        dtype=int,
    )

    tn = int(
        np.sum(
            (truth == 0)
            &
            (prediction == 0)
        )
    )

    fp = int(
        np.sum(
            (truth == 0)
            &
            (prediction == 1)
        )
    )

    fn = int(
        np.sum(
            (truth == 1)
            &
            (prediction == 0)
        )
    )

    tp = int(
        np.sum(
            (truth == 1)
            &
            (prediction == 1)
        )
    )

    return np.asarray(
        [
            tn,
            fp,
            fn,
            tp,
        ],
        dtype=np.int64,
    )


def metrics_from_counts(
    counts,
):
    """
    counts can be:
      shape (4,)
      or shape (R,4)

    order:
      TN, FP, FN, TP
    """

    counts = np.asarray(
        counts,
        dtype=float,
    )

    tn = counts[..., 0]
    fp = counts[..., 1]
    fn = counts[..., 2]
    tp = counts[..., 3]

    total = (
        tn
        + fp
        + fn
        + tp
    )

    accuracy = np.divide(
        tn + tp,
        total,
        out=np.zeros_like(
            total,
            dtype=float,
        ),
        where=(
            total != 0
        ),
    )

    denom_0 = (
        2 * tn
        + fp
        + fn
    )

    denom_1 = (
        2 * tp
        + fp
        + fn
    )

    f1_0 = np.divide(
        2 * tn,
        denom_0,
        out=np.zeros_like(
            denom_0,
            dtype=float,
        ),
        where=(
            denom_0 != 0
        ),
    )

    f1_1 = np.divide(
        2 * tp,
        denom_1,
        out=np.zeros_like(
            denom_1,
            dtype=float,
        ),
        where=(
            denom_1 != 0
        ),
    )

    macro_f1 = (
        f1_0
        + f1_1
    ) / 2.0

    return (
        accuracy,
        macro_f1,
    )


# =====================================================================
# 10. BUILD PER-MATCH CONFUSION CONTRIBUTIONS
# =====================================================================

truth = wide[
    "true_label"
].to_numpy(
    dtype=int
)


model_prediction_arrays = {}

model_match_count_contributions = {}


for model_name in sorted(
    expected_models
):

    safe_name = (
        model_name
        .replace(
            "-",
            "_"
        )
    )

    pred = wide[
        f"pred_{safe_name}"
    ].to_numpy(
        dtype=int
    )

    model_prediction_arrays[
        model_name
    ] = pred

    contributions = np.zeros(
        (
            len(
                wide
            ),
            4,
        ),
        dtype=np.int16,
    )

    contributions[
        (
            truth == 0
        )
        &
        (
            pred == 0
        ),
        0,
    ] = 1

    contributions[
        (
            truth == 0
        )
        &
        (
            pred == 1
        ),
        1,
    ] = 1

    contributions[
        (
            truth == 1
        )
        &
        (
            pred == 0
        ),
        2,
    ] = 1

    contributions[
        (
            truth == 1
        )
        &
        (
            pred == 1
        ),
        3,
    ] = 1

    model_match_count_contributions[
        model_name
    ] = contributions


# =====================================================================
# 11. POINT ESTIMATES
# =====================================================================

point_metrics = {}


for model_name in sorted(
    expected_models
):

    counts = (
        model_match_count_contributions[
            model_name
        ].sum(
            axis=0
        )
    )

    accuracy, macro_f1 = (
        metrics_from_counts(
            counts
        )
    )

    point_metrics[
        model_name
    ] = {
        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "tn":
            int(
                counts[0]
            ),

        "fp":
            int(
                counts[1]
            ),

        "fn":
            int(
                counts[2]
            ),

        "tp":
            int(
                counts[3]
            ),
    }


point_metrics_df = pd.DataFrame(
    [
        {
            "model":
                model_name,

            **values,
        }

        for (
            model_name,
            values
        )
        in point_metrics.items()
    ]
)


# =====================================================================
# 12. MATCH-LEVEL PAIRED BOOTSTRAP — REFERENCE ONLY
# =====================================================================
#
# This recreates the original iid-style resampling assumption.
# It is included so the dependency-aware intervals can be compared
# directly against the ordinary match-level intervals.
#
# Batched to keep Colab memory usage modest.
# =====================================================================

def match_level_bootstrap_for_comparison(
    model_a,
    model_b,
    n_resamples,
    seed,
    batch_size=250,
):

    rng = np.random.default_rng(
        seed
    )

    n = len(
        wide
    )

    contributions_a = (
        model_match_count_contributions[
            model_a
        ]
    )

    contributions_b = (
        model_match_count_contributions[
            model_b
        ]
    )

    accuracy_differences = np.empty(
        n_resamples,
        dtype=np.float64,
    )

    macro_f1_differences = np.empty(
        n_resamples,
        dtype=np.float64,
    )

    cursor = 0

    while cursor < n_resamples:

        current_batch = min(
            batch_size,
            n_resamples - cursor,
        )

        sample_indices = rng.integers(
            0,
            n,
            size=(
                current_batch,
                n,
            ),
            dtype=np.int32,
        )

        counts_a = (
            contributions_a[
                sample_indices
            ]
            .sum(
                axis=1
            )
        )

        counts_b = (
            contributions_b[
                sample_indices
            ]
            .sum(
                axis=1
            )
        )

        (
            acc_a,
            f1_a,
        ) = metrics_from_counts(
            counts_a
        )

        (
            acc_b,
            f1_b,
        ) = metrics_from_counts(
            counts_b
        )

        accuracy_differences[
            cursor:
            cursor + current_batch
        ] = (
            acc_a
            - acc_b
        )

        macro_f1_differences[
            cursor:
            cursor + current_batch
        ] = (
            f1_a
            - f1_b
        )

        cursor += current_batch

    return (
        accuracy_differences,
        macro_f1_differences,
    )


# =====================================================================
# 13. CLUSTER-LEVEL CONFUSION COUNTS
# =====================================================================

def build_cluster_counts(
    cluster_column,
):

    cluster_ids = (
        wide[
            cluster_column
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    cluster_index = {
        cluster_id:
            index

        for (
            index,
            cluster_id
        )
        in enumerate(
            cluster_ids
        )
    }

    block_indices = np.asarray(
        [
            cluster_index[
                value
            ]
            for value in wide[
                cluster_column
            ]
        ],
        dtype=int,
    )

    model_counts = {}

    for model_name in sorted(
        expected_models
    ):

        counts = np.zeros(
            (
                len(
                    cluster_ids
                ),
                4,
            ),
            dtype=np.int64,
        )

        contributions = (
            model_match_count_contributions[
                model_name
            ]
        )

        for row_position, cluster_position in enumerate(
            block_indices
        ):

            counts[
                cluster_position
            ] += contributions[
                row_position
            ]

        model_counts[
            model_name
        ] = counts

    return (
        cluster_ids,
        model_counts,
    )


(
    comp_season_ids,
    comp_season_model_counts,
) = build_cluster_counts(
    "competition_season_block"
)


(
    competition_ids,
    competition_model_counts,
) = build_cluster_counts(
    "competition_block"
)


# =====================================================================
# 14. CLUSTER BOOTSTRAP
# =====================================================================
#
# Whole clusters are sampled WITH REPLACEMENT.
#
# If a competition-season is selected twice, every match in that
# competition-season appears twice in the bootstrap replicate.
#
# The SAME sampled clusters are used for both models in a comparison.
#
# This preserves pairedness.
# =====================================================================

def cluster_bootstrap_all_models(
    cluster_ids,
    model_counts,
    n_resamples,
    seed,
):

    rng = np.random.default_rng(
        seed
    )

    k = len(
        cluster_ids
    )

    sampled_cluster_positions = rng.integers(
        0,
        k,
        size=(
            n_resamples,
            k,
        ),
        dtype=np.int16,
    )

    bootstrap_metrics = {}

    for model_name in sorted(
        expected_models
    ):

        counts = model_counts[
            model_name
        ]

        bootstrap_counts = (
            counts[
                sampled_cluster_positions
            ]
            .sum(
                axis=1
            )
        )

        (
            accuracy,
            macro_f1,
        ) = metrics_from_counts(
            bootstrap_counts
        )

        bootstrap_metrics[
            model_name
        ] = {
            "accuracy":
                accuracy,

            "macro_f1":
                macro_f1,
        }

    return (
        sampled_cluster_positions,
        bootstrap_metrics,
    )


(
    comp_season_draws,
    comp_season_bootstrap,
) = cluster_bootstrap_all_models(
    comp_season_ids,
    comp_season_model_counts,
    N_BOOTSTRAP,
    SEED_COMP_SEASON_BOOTSTRAP,
)


(
    competition_draws,
    competition_bootstrap,
) = cluster_bootstrap_all_models(
    competition_ids,
    competition_model_counts,
    N_BOOTSTRAP,
    SEED_COMPETITION_BOOTSTRAP,
)


# =====================================================================
# 15. BOOTSTRAP INTERVAL HELPER
# =====================================================================

def percentile_interval(
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    return (
        float(
            np.percentile(
                values,
                2.5,
            )
        ),

        float(
            np.percentile(
                values,
                97.5,
            )
        ),
    )


def summarize_difference_distribution(
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    lower, upper = (
        percentile_interval(
            values
        )
    )

    return {
        "mean_difference_pp":
            100.0
            * float(
                values.mean()
            ),

        "median_difference_pp":
            100.0
            * float(
                np.median(
                    values
                )
            ),

        "ci_lower_pp":
            100.0
            * lower,

        "ci_upper_pp":
            100.0
            * upper,

        "proportion_difference_le_zero":
            float(
                np.mean(
                    values <= 0
                )
            ),

        "proportion_difference_ge_zero":
            float(
                np.mean(
                    values >= 0
                )
            ),
    }


# =====================================================================
# 16. RUN REFERENCE MATCH BOOTSTRAPS
# =====================================================================

match_bootstrap_results = {}


for comparison_number, (
    model_a,
    model_b,
    label,
    comparison_role,
) in enumerate(
    COMPARISONS
):

    print(
        "\nRunning reference match-level bootstrap:"
    )

    print(
        label
    )

    (
        match_acc_delta,
        match_f1_delta,
    ) = match_level_bootstrap_for_comparison(
        model_a,
        model_b,
        N_BOOTSTRAP,
        SEED_MATCH_BOOTSTRAP
        + comparison_number,
    )

    match_bootstrap_results[
        label
    ] = {
        "accuracy_delta":
            match_acc_delta,

        "macro_f1_delta":
            match_f1_delta,
    }


# =====================================================================
# 17. SUMMARIZE ALL THREE INFERENCE ASSUMPTIONS
# =====================================================================

summary_rows = []

replicate_rows_comp_season = []
replicate_rows_competition = []


for (
    model_a,
    model_b,
    label,
    comparison_role,
) in COMPARISONS:

    # -------------------------------------------------------------
    # POINT DIFFERENCES
    # -------------------------------------------------------------

    point_accuracy_difference = (
        point_metrics[
            model_a
        ][
            "accuracy"
        ]
        -
        point_metrics[
            model_b
        ][
            "accuracy"
        ]
    )

    point_f1_difference = (
        point_metrics[
            model_a
        ][
            "macro_f1"
        ]
        -
        point_metrics[
            model_b
        ][
            "macro_f1"
        ]
    )

    # -------------------------------------------------------------
    # MATCH-LEVEL REFERENCE
    # -------------------------------------------------------------

    iid_acc = (
        match_bootstrap_results[
            label
        ][
            "accuracy_delta"
        ]
    )

    iid_f1 = (
        match_bootstrap_results[
            label
        ][
            "macro_f1_delta"
        ]
    )

    (
        iid_acc_lower,
        iid_acc_upper,
    ) = percentile_interval(
        iid_acc
    )

    (
        iid_f1_lower,
        iid_f1_upper,
    ) = percentile_interval(
        iid_f1
    )

    # -------------------------------------------------------------
    # COMPETITION-SEASON BOOTSTRAP
    # -------------------------------------------------------------

    cs_acc = (
        comp_season_bootstrap[
            model_a
        ][
            "accuracy"
        ]
        -
        comp_season_bootstrap[
            model_b
        ][
            "accuracy"
        ]
    )

    cs_f1 = (
        comp_season_bootstrap[
            model_a
        ][
            "macro_f1"
        ]
        -
        comp_season_bootstrap[
            model_b
        ][
            "macro_f1"
        ]
    )

    (
        cs_acc_lower,
        cs_acc_upper,
    ) = percentile_interval(
        cs_acc
    )

    (
        cs_f1_lower,
        cs_f1_upper,
    ) = percentile_interval(
        cs_f1
    )

    # -------------------------------------------------------------
    # COMPETITION-LEVEL BOOTSTRAP
    # -------------------------------------------------------------

    comp_acc = (
        competition_bootstrap[
            model_a
        ][
            "accuracy"
        ]
        -
        competition_bootstrap[
            model_b
        ][
            "accuracy"
        ]
    )

    comp_f1 = (
        competition_bootstrap[
            model_a
        ][
            "macro_f1"
        ]
        -
        competition_bootstrap[
            model_b
        ][
            "macro_f1"
        ]
    )

    (
        comp_acc_lower,
        comp_acc_upper,
    ) = percentile_interval(
        comp_acc
    )

    (
        comp_f1_lower,
        comp_f1_upper,
    ) = percentile_interval(
        comp_f1
    )

    # -------------------------------------------------------------
    # SUMMARY ROW
    # -------------------------------------------------------------

    summary_rows.append({
        "comparison":
            label,

        "comparison_role":
            comparison_role,

        "model_a":
            model_a,

        "model_b":
            model_b,

        "paired_n":
            EXPECTED_PAIRED_N,

        "point_model_a_macro_f1":
            point_metrics[
                model_a
            ][
                "macro_f1"
            ],

        "point_model_b_macro_f1":
            point_metrics[
                model_b
            ][
                "macro_f1"
            ],

        "point_macro_f1_difference_pp":
            100.0
            * point_f1_difference,

        "point_accuracy_difference_pp":
            100.0
            * point_accuracy_difference,

        # Match-level reference
        "match_bootstrap_macro_f1_ci_lower_pp":
            100.0
            * iid_f1_lower,

        "match_bootstrap_macro_f1_ci_upper_pp":
            100.0
            * iid_f1_upper,

        "match_bootstrap_accuracy_ci_lower_pp":
            100.0
            * iid_acc_lower,

        "match_bootstrap_accuracy_ci_upper_pp":
            100.0
            * iid_acc_upper,

        # Primary dependency-aware
        "competition_season_blocks":
            len(
                comp_season_ids
            ),

        "comp_season_macro_f1_ci_lower_pp":
            100.0
            * cs_f1_lower,

        "comp_season_macro_f1_ci_upper_pp":
            100.0
            * cs_f1_upper,

        "comp_season_accuracy_ci_lower_pp":
            100.0
            * cs_acc_lower,

        "comp_season_accuracy_ci_upper_pp":
            100.0
            * cs_acc_upper,

        "comp_season_probability_delta_le_zero":
            float(
                np.mean(
                    cs_f1 <= 0
                )
            ),

        # Coarser sensitivity
        "competition_blocks":
            len(
                competition_ids
            ),

        "competition_macro_f1_ci_lower_pp":
            100.0
            * comp_f1_lower,

        "competition_macro_f1_ci_upper_pp":
            100.0
            * comp_f1_upper,

        "competition_accuracy_ci_lower_pp":
            100.0
            * comp_acc_lower,

        "competition_accuracy_ci_upper_pp":
            100.0
            * comp_acc_upper,

        "competition_probability_delta_le_zero":
            float(
                np.mean(
                    comp_f1 <= 0
                )
            ),
    })

    # Save full dependency-aware replicate deltas for audit.
    for replicate in range(
        N_BOOTSTRAP
    ):

        replicate_rows_comp_season.append({
            "comparison":
                label,

            "replicate":
                replicate + 1,

            "accuracy_difference_pp":
                100.0
                * float(
                    cs_acc[
                        replicate
                    ]
                ),

            "macro_f1_difference_pp":
                100.0
                * float(
                    cs_f1[
                        replicate
                    ]
                ),
        })

        replicate_rows_competition.append({
            "comparison":
                label,

            "replicate":
                replicate + 1,

            "accuracy_difference_pp":
                100.0
                * float(
                    comp_acc[
                        replicate
                    ]
                ),

            "macro_f1_difference_pp":
                100.0
                * float(
                    comp_f1[
                        replicate
                    ]
                ),
        })


summary_df = pd.DataFrame(
    summary_rows
)


# =====================================================================
# 18. BUILD BLOCK MANIFESTS
# =====================================================================

comp_season_manifest = (
    metadata
    .groupby(
        [
            "competition_id",
            "competition_name",
            "season_id",
            "competition_season_block",
        ],
        dropna=False,
    )
    .agg(
        n_matches=(
            "match_id",
            "size",
        ),

        n_win=(
            "true_label",
            "sum",
        ),
    )
    .reset_index()
)


comp_season_manifest[
    "n_not_win"
] = (
    comp_season_manifest[
        "n_matches"
    ]
    -
    comp_season_manifest[
        "n_win"
    ]
)


comp_season_manifest[
    "win_rate"
] = (
    comp_season_manifest[
        "n_win"
    ]
    /
    comp_season_manifest[
        "n_matches"
    ]
)


competition_manifest = (
    metadata
    .groupby(
        [
            "competition_id",
            "competition_name",
            "competition_block",
        ],
        dropna=False,
    )
    .agg(
        n_matches=(
            "match_id",
            "size",
        ),

        n_seasons=(
            "season_id",
            "nunique",
        ),

        n_win=(
            "true_label",
            "sum",
        ),
    )
    .reset_index()
)


competition_manifest[
    "n_not_win"
] = (
    competition_manifest[
        "n_matches"
    ]
    -
    competition_manifest[
        "n_win"
    ]
)


competition_manifest[
    "win_rate"
] = (
    competition_manifest[
        "n_win"
    ]
    /
    competition_manifest[
        "n_matches"
    ]
)


# =====================================================================
# 19. LEAVE-ONE-BLOCK-OUT SENSITIVITY
# =====================================================================
#
# This is not the primary inferential method.
# It is an influence diagnostic:
# "Does one competition-season or one competition determine the result?"
# =====================================================================

def leave_one_cluster_out(
    cluster_column,
    comparisons,
):

    cluster_values = (
        wide[
            cluster_column
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    output_rows = []

    for omitted_cluster in cluster_values:

        keep_mask = (
            wide[
                cluster_column
            ]
            != omitted_cluster
        )

        y_keep = (
            wide.loc[
                keep_mask,
                "true_label",
            ]
            .to_numpy(
                dtype=int
            )
        )

        for (
            model_a,
            model_b,
            label,
            comparison_role,
        ) in comparisons:

            safe_a = (
                model_a
                .replace(
                    "-",
                    "_"
                )
            )

            safe_b = (
                model_b
                .replace(
                    "-",
                    "_"
                )
            )

            pred_a = (
                wide.loc[
                    keep_mask,
                    f"pred_{safe_a}",
                ]
                .to_numpy(
                    dtype=int
                )
            )

            pred_b = (
                wide.loc[
                    keep_mask,
                    f"pred_{safe_b}",
                ]
                .to_numpy(
                    dtype=int
                )
            )

            counts_a = confusion_counts(
                y_keep,
                pred_a,
            )

            counts_b = confusion_counts(
                y_keep,
                pred_b,
            )

            (
                acc_a,
                f1_a,
            ) = metrics_from_counts(
                counts_a
            )

            (
                acc_b,
                f1_b,
            ) = metrics_from_counts(
                counts_b
            )

            output_rows.append({
                "cluster_type":
                    cluster_column,

                "omitted_cluster":
                    omitted_cluster,

                "comparison":
                    label,

                "comparison_role":
                    comparison_role,

                "remaining_n":
                    int(
                        keep_mask.sum()
                    ),

                "macro_f1_difference_pp":
                    100.0
                    * float(
                        f1_a
                        - f1_b
                    ),

                "accuracy_difference_pp":
                    100.0
                    * float(
                        acc_a
                        - acc_b
                    ),
            })

    return pd.DataFrame(
        output_rows
    )


leave_one_comp_season = (
    leave_one_cluster_out(
        "competition_season_block",
        COMPARISONS,
    )
)


leave_one_competition = (
    leave_one_cluster_out(
        "competition_block",
        COMPARISONS,
    )
)


# =====================================================================
# 20. CREATE INFLUENCE SUMMARY
# =====================================================================

influence_rows = []


for label in [
    comparison[
        2
    ]
    for comparison in COMPARISONS
]:

    block_cs = (
        leave_one_comp_season[
            leave_one_comp_season[
                "comparison"
            ].eq(
                label
            )
        ]
    )

    block_comp = (
        leave_one_competition[
            leave_one_competition[
                "comparison"
            ].eq(
                label
            )
        ]
    )

    influence_rows.append({
        "comparison":
            label,

        "leave_one_comp_season_min_delta_pp":
            float(
                block_cs[
                    "macro_f1_difference_pp"
                ].min()
            ),

        "leave_one_comp_season_max_delta_pp":
            float(
                block_cs[
                    "macro_f1_difference_pp"
                ].max()
            ),

        "leave_one_competition_min_delta_pp":
            float(
                block_comp[
                    "macro_f1_difference_pp"
                ].min()
            ),

        "leave_one_competition_max_delta_pp":
            float(
                block_comp[
                    "macro_f1_difference_pp"
                ].max()
            ),
    })


influence_summary = pd.DataFrame(
    influence_rows
)


# =====================================================================
# 21. SAVE ALL OUTPUTS
# =====================================================================

RESULT_DIR = (
    OUTPUT_ROOT
    / "outputs"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


summary_df.to_csv(
    RESULT_DIR
    / "dependency_aware_summary.csv",
    index=False,
)


point_metrics_df.to_csv(
    RESULT_DIR
    / "point_model_metrics.csv",
    index=False,
)


comp_season_manifest.to_csv(
    RESULT_DIR
    / "competition_season_block_manifest.csv",
    index=False,
)


competition_manifest.to_csv(
    RESULT_DIR
    / "competition_block_manifest.csv",
    index=False,
)


pd.DataFrame(
    replicate_rows_comp_season
).to_csv(
    RESULT_DIR
    / "competition_season_bootstrap_replicates.csv",
    index=False,
)


pd.DataFrame(
    replicate_rows_competition
).to_csv(
    RESULT_DIR
    / "competition_bootstrap_replicates.csv",
    index=False,
)


leave_one_comp_season.to_csv(
    RESULT_DIR
    / "leave_one_competition_season_out.csv",
    index=False,
)


leave_one_competition.to_csv(
    RESULT_DIR
    / "leave_one_competition_out.csv",
    index=False,
)


influence_summary.to_csv(
    RESULT_DIR
    / "block_influence_summary.csv",
    index=False,
)


# =====================================================================
# 22. FROZEN PROTOCOL RECORD
# =====================================================================

protocol_record = {
    "phase":
        "Phase 3 - dependency-aware inference",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "input_artifact":
        EXPECTED_INPUT_NAME,

    "input_combined_predictions_sha256":
        combined_sha,

    "paired_oof_n":
        EXPECTED_PAIRED_N,

    "primary_metric":
        "macro F1 difference",

    "reference_inference":
        (
            "paired individual-match bootstrap "
            "(iid-style reference only)"
        ),

    "primary_dependency_aware_method":
        (
            "paired competition-season cluster bootstrap"
        ),

    "primary_block_definition":
        (
            "competition_id x season_id"
        ),

    "primary_block_count":
        int(
            n_comp_season_blocks
        ),

    "secondary_dependency_aware_method":
        (
            "paired competition-level cluster bootstrap"
        ),

    "secondary_block_definition":
        "competition_id",

    "secondary_block_count":
        int(
            n_competition_blocks
        ),

    "bootstrap_resamples":
        N_BOOTSTRAP,

    "cluster_resampling_rule":
        (
            "sample whole clusters with replacement; "
            "include all matches in every selected cluster; "
            "use the same sampled clusters for both models "
            "within each paired comparison"
        ),

    "confidence_interval":
        "percentile 95% bootstrap interval",

    "comparisons": [
        {
            "model_a":
                a,

            "model_b":
                b,

            "label":
                label,

            "role":
                role,
        }

        for (
            a,
            b,
            label,
            role,
        ) in COMPARISONS
    ],

    "interpretation_rule":
        (
            "If dependency-aware intervals are wider or cross "
            "zero when the match-level interval does not, "
            "report that inferential support is sensitive to "
            "the assumed dependence structure. Do not hide "
            "or replace the dependency-aware result."
        ),

    "limitation":
        (
            "Cluster bootstrap addresses within-block dependence "
            "but cannot model every possible cross-season or "
            "cross-competition dependency."
        ),
}


(
    RESULT_DIR
    / "phase3_protocol.json"
).write_text(
    json.dumps(
        protocol_record,
        indent=2,
    ),
    encoding="utf-8",
)


# =====================================================================
# 23. VALIDATION RECORD
# =====================================================================

validation_record = {
    "input_hash_verified":
        True,

    "paired_oof_verified":
        True,

    "paired_n":
        int(
            len(
                reference_ids
            )
        ),

    "model_count":
        int(
            len(
                actual_models
            )
        ),

    "competition_season_blocks":
        int(
            n_comp_season_blocks
        ),

    "competition_blocks":
        int(
            n_competition_blocks
        ),

    "bootstrap_resamples":
        N_BOOTSTRAP,

    "no_model_retraining":
        True,

    "no_prediction_changes":
        True,
}


(
    RESULT_DIR
    / "phase3_validation.json"
).write_text(
    json.dumps(
        validation_record,
        indent=2,
    ),
    encoding="utf-8",
)


# =====================================================================
# 24. CREATE HUMAN-READABLE README
# =====================================================================

readme_text = f"""
IEEE ACCESS R1 — PHASE 3
DEPENDENCY-AWARE STATISTICAL INFERENCE
======================================

Purpose
-------
Assess whether the Global paired performance conclusions remain stable
when dependence between matches is acknowledged through block resampling.

Frozen data
-----------
Paired Global OOF observations: {EXPECTED_PAIRED_N}

No neural model was retrained and no prediction was modified.

Primary sensitivity analysis
----------------------------
Competition-season cluster bootstrap.

Block definition:
competition_id x season_id

Number of blocks:
{n_comp_season_blocks}

Bootstrap replicates:
{N_BOOTSTRAP}

Whole competition-season blocks are sampled with replacement.
All matches belonging to every sampled block are retained.
The same sampled blocks are used for both models in each paired comparison.

Secondary coarser sensitivity
-----------------------------
Competition-level cluster bootstrap.

Number of competition blocks:
{n_competition_blocks}

Reference analysis
------------------
Individual-match paired bootstrap, retained only to show how the
uncertainty interval changes when dependency assumptions are relaxed.

Primary comparison
------------------
C-Hybrid-full vs A-MLP.

Additional comparisons
----------------------
C-Hybrid-full vs TF-Hybrid
TF-Hybrid vs A-MLP
B-GNN-full vs TF-NN

Interpretation
--------------
If the block-bootstrap interval becomes wider or crosses zero while
the individual-match interval does not, the paper must explicitly state
that inferential support is sensitive to the assumed dependence structure.

The block bootstrap is a sensitivity analysis rather than proof that all
dependency has been eliminated. Cross-season and other forms of residual
dependence may remain.
""".strip()


(
    RESULT_DIR
    / "README_PHASE3.txt"
).write_text(
    readme_text,
    encoding="utf-8",
)


# =====================================================================
# 25. PRINT RESULTS
# =====================================================================

display_columns = [
    "comparison",
    "point_macro_f1_difference_pp",

    "match_bootstrap_macro_f1_ci_lower_pp",
    "match_bootstrap_macro_f1_ci_upper_pp",

    "comp_season_macro_f1_ci_lower_pp",
    "comp_season_macro_f1_ci_upper_pp",

    "competition_macro_f1_ci_lower_pp",
    "competition_macro_f1_ci_upper_pp",
]


print("\n" + "=" * 120)
print("PHASE 3 RESULTS")
print("=" * 120)


with pd.option_context(
    "display.max_columns",
    None,
    "display.width",
    250,
):

    print(
        summary_df[
            display_columns
        ].to_string(
            index=False
        )
    )


print("\n" + "=" * 120)
print("BLOCK STRUCTURE")
print("=" * 120)


print(
    "\nCompetition-season blocks:",
    n_comp_season_blocks
)

print(
    "Competition blocks:",
    n_competition_blocks
)


print(
    "\nCompetition-season block sizes:"
)

print(
    comp_season_manifest[
        [
            "competition_name",
            "competition_id",
            "season_id",
            "n_matches",
            "n_win",
            "n_not_win",
            "win_rate",
        ]
    ]
    .sort_values(
        [
            "competition_name",
            "season_id",
        ]
    )
    .to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("LEAVE-ONE-BLOCK-OUT INFLUENCE RANGE")
print("=" * 120)


print(
    influence_summary.to_string(
        index=False
    )
)


# =====================================================================
# 26. AUTOMATIC INTERPRETATION FLAGS
# =====================================================================

print("\n" + "=" * 120)
print("AUTOMATIC INTERPRETATION FLAGS")
print("=" * 120)


for _, row in summary_df.iterrows():

    label = row[
        "comparison"
    ]

    point = row[
        "point_macro_f1_difference_pp"
    ]

    iid_low = row[
        "match_bootstrap_macro_f1_ci_lower_pp"
    ]

    iid_high = row[
        "match_bootstrap_macro_f1_ci_upper_pp"
    ]

    cs_low = row[
        "comp_season_macro_f1_ci_lower_pp"
    ]

    cs_high = row[
        "comp_season_macro_f1_ci_upper_pp"
    ]

    comp_low = row[
        "competition_macro_f1_ci_lower_pp"
    ]

    comp_high = row[
        "competition_macro_f1_ci_upper_pp"
    ]

    print(
        f"\n{label}"
    )

    print(
        f"  Point difference: {point:+.3f} pp"
    )

    print(
        f"  Match bootstrap: "
        f"[{iid_low:+.3f}, {iid_high:+.3f}] pp"
    )

    print(
        f"  Competition-season bootstrap: "
        f"[{cs_low:+.3f}, {cs_high:+.3f}] pp"
    )

    print(
        f"  Competition bootstrap: "
        f"[{comp_low:+.3f}, {comp_high:+.3f}] pp"
    )

    if (
        iid_low > 0
        and
        cs_low <= 0 <= cs_high
    ):

        print(
            "  IMPORTANT: match-level support disappears "
            "under competition-season block resampling."
        )

    elif (
        cs_low > 0
    ):

        print(
            "  Competition-season sensitivity remains "
            "entirely above zero."
        )

    else:

        print(
            "  Competition-season analysis does not "
            "establish a positive difference."
        )

    if (
        comp_low <= 0 <= comp_high
    ):

        print(
            "  Coarse competition-level sensitivity "
            "includes zero."
        )

    elif (
        comp_low > 0
    ):

        print(
            "  Coarse competition-level sensitivity "
            "also remains above zero."
        )


# =====================================================================
# 27. PACKAGE RESULTS
# =====================================================================

if FINAL_ZIP.exists():

    FINAL_ZIP.unlink()


with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(
        RESULT_DIR.rglob(
            "*"
        )
    ):

        if path.is_file():

            archive.write(
                path,
                Path(
                    "outputs"
                )
                / path.relative_to(
                    RESULT_DIR
                ),
            )


print("\n" + "=" * 100)
print("PHASE 3 COMPLETE")
print("=" * 100)

print(
    "\nOutput ZIP:"
)

print(
    FINAL_ZIP
)

print(
    "\nUpload this ZIP to ChatGPT before moving to Phase 4."
)


# =====================================================================
# 28. AUTOMATIC DOWNLOAD
# =====================================================================

try:

    from google.colab import files

    print(
        "\nStarting automatic download..."
    )

    files.download(
        str(
            FINAL_ZIP
        )
    )

except Exception as exc:

    print(
        "\nAutomatic download did not start:"
    )

    print(
        exc
    )

    print(
        "\nDownload manually from the Colab Files panel:"
    )

    print(
        FINAL_ZIP
    )

IEEE ACCESS R1 — PHASE 3
DEPENDENCY-AWARE GLOBAL STATISTICAL INFERENCE

No GPU is required. This phase uses frozen OOF predictions only.

Phase 2B ZIP was not found in /content/.
Please upload:
IEEE_Access_R1_Exp6_Topology_Free_GLOBAL.zip


Saving IEEE_Access_R1_Exp6_Topology_Free_GLOBAL.zip to IEEE_Access_R1_Exp6_Topology_Free_GLOBAL.zip

Input ZIP: /content/IEEE_Access_R1_Exp6_Topology_Free_GLOBAL.zip

Combined prediction SHA-256:
d5df0e00932a375dc7faa998778c85d944436d43267257c269a225355c689aab
✓ Exact frozen Phase 2B predictions verified.

✓ Exact paired OOF set verified.
Paired N: 1745

Competition-season blocks: 20
Competition blocks: 11
✓ Dependency block structure verified.

Running reference match-level bootstrap:
C-Hybrid-full vs A-MLP

Running reference match-level bootstrap:
C-Hybrid-full vs TF-Hybrid

Running reference match-level bootstrap:
TF-Hybrid vs A-MLP

Running reference match-level bootstrap:
B-GNN-full vs TF-NN

PHASE 3 RESULTS
                comparison  point_macro_f1_difference_pp  match_bootstrap_macro_f1_ci_lower_pp  match_bootstrap_macro_f1_ci_upper_pp  comp_season_macro_f1_ci_lower_pp  comp_season_macro_f1_ci_upper_pp  competition_macro_f1_ci_lower_pp  competition_macro_f1_ci_upper_pp
    C-Hy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>